# Task 5: Model Evaluation and Kaggle Submission

## 5.1 Validation Set Evaluation

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
from torchvision.models import resnet18
from data_pipeline import create_dataloaders

PROJECT_DIR = Path.cwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the same validation dataset used during training
data = create_dataloaders(data_dir=PROJECT_DIR)

val_loader = data.val_loader
class_to_idx = data.class_to_idx
idx_to_class = data.idx_to_class

# Load the best fine-tuned checkpoint
checkpoint = torch.load(
    PROJECT_DIR / "best_resnet18_finetuned.pth",
    map_location=device,
    weights_only=True
)

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 8)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

assert checkpoint["class_to_idx"] == class_to_idx

print("Device:", device)
print("Validation images:", len(val_loader.dataset))
print("Best fine-tuning epoch:", checkpoint["epoch"])
print(f"Saved validation accuracy: {checkpoint['val_accuracy'] * 100:.2f}%")